In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype


In [6]:
# Note for future universality: a new data file can be easily added in the file_months step below
    # and the rest of the code ran to continually use and update the data file used in Tableau  


# File dictionary: update this as new months are added
file_months = {"April 2024.xlsx": "April",
               "May 2024.xlsx": "May",
               "June 2024.xlsx": "June",
               "July 2024.xlsx": "July",
               "August 2024.xlsx": "August",
               "September 2024.xlsx": "September",
               "October 2024.xlsx": "October",
               "November 2024 corrected.xlsx": "November",
               "December 2024.xlsx": "December",
               "January 2025.xlsx": "January",
               "February 2025.xlsx": "February",
               "March 2025.xlsx": "March"
    
}


In [8]:
all_data = []
for file, month in file_months.items():
    df_month = pd.read_excel(file, engine="openpyxl")
    df_month['Start time'] = pd.to_datetime(df_month['Start time'])
    df_month['Month_Day'] = df_month['Start time'].dt.strftime('%B, %-d')
    all_data.append(df_month)


In [89]:
df = pd.concat(all_data, ignore_index=True)


In [91]:
df = df.drop_duplicates()  


In [93]:
intake_line_numbers = {
    "A2J Immigration": 13123478347,
    "A2J Immigration Toll Free": 18882652188,
    "Austin Intake VM": 13124235904,
    "Bankruptcy Helpdesk VM": 13122296344, 
    "CLASP VM": 13124235900,
    "Criminal Records": 13122296071,
    "Education Law Referrals VM": 13123478392,
    "Fair Housing Intake VM": 13124235909, 
    "HIV Intake VM": 13123478309,
    "JEHD": 13122296072,
    "legalclinics": 13124235938,
    "OP Appeals Project": 13124312101, 
    "Veterans Rights Project VM": 13123478340,
    "Trafficking Survivors Assistance Project": 13122296073,
    "Migrant Legal Assistance Program": 13124312299,
    "Nursing Home Ombudsman": 13122296079,
    "Markham Eviction Help Desk": 13122296014
}

In [95]:
special_intake_numbers = set(intake_line_numbers.values())
main_number = 13123411070

# Mapping number to name
number_to_name = {v: k for k, v in intake_line_numbers.items()}

df['Intake Line Name'] = df['Called number'].map(number_to_name)

# Adding label for main number
df.loc[df['Called number'] == main_number, 'Intake Line Name'] = 'Main Number'

# -- Line Type Classification --
# Adding Line Type column
def classify_line_type(number):
    if pd.isna(number):
        return "Unknown"
    elif number == main_number:
        return "Main Number"
    elif number in special_intake_numbers:
        return "Special Intake Lines"
    else:
        return "Non-Special Intake Lines"

df['Line Type'] = df['Called number'].apply(classify_line_type)

In [97]:
df.sort_values(by=['Correlation ID', 'Start time'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()


In [103]:
# Adding many extra columns and time/date features for ease of Tableau visualization
df['Hour'] = df['Start time'].dt.hour
df['Start time (Hour/Min)'] = df['Start time'].dt.strftime('%I:%M %p')
df['Month_Day'] = df['Start time'].dt.strftime('%B %-d')
df['Day of week'] = df['Start time'].dt.day_name()
df['Month_Year'] = df['Start time'].dt.strftime('%B %Y')
df['Date only'] = df['Start time'].dt.date

# Adding Weekend column
df['Weekend'] = df['Day of week'].apply(
    lambda day: 'Weekend' if day in ['Saturday', 'Sunday'] else 'Weekday'
)

df['Business hours'] = df['Start time'].apply(
    lambda t: 'Business Hours' if pd.to_datetime('08:00:00').time() <= t.time() <= pd.to_datetime('17:00:00').time()
    else 'Outside Business Hours'
)


# Adding time bucket labels
hour_labels = [
    f"{(h % 12 or 12)}:00 {'AM' if h < 12 else 'PM'} - {(h % 12 or 12)}:59 {'AM' if h < 12 else 'PM'}"
    for h in range(24)
]

df['Time bucket'] = df['Hour'].apply(
    lambda h: f"{(h % 12 or 12)}:00 {'AM' if h < 12 else 'PM'} - {(h % 12 or 12)}:59 {'AM' if h < 12 else 'PM'}"
)

# Setting as ordered categorical for Tableau-friendly sorting
time_bucket_type = CategoricalDtype(categories=hour_labels, ordered=True)
df['Time bucket'] = df['Time bucket'].astype(time_bucket_type)

# Function to classify call types
def determine_call_type(row):
    if pd.isna(row['PSTN vendor name']):
        return 'Internal Transfer'
    elif row['PSTN vendor name'] == 'CallTower' and row['Direction'] == 'TERMINATING':
        return 'Inbound'
    elif row['PSTN vendor name'] == 'CallTower' and row['Direction'] == 'ORIGINATING':
        return 'Outbound'
    else:
        return 'Unknown'

df['Prelim Call Type'] = df.apply(determine_call_type, axis=1)

# Assigning call type and deduplicating internal transfers
df['Call Type'] = df['Prelim Call Type']  

internal_transfer_mask = df['Prelim Call Type'] == 'Internal Transfer'

# marking only the first instance of internal transfer call
df.loc[internal_transfer_mask, 'Transfer Group'] = df[internal_transfer_mask].groupby(
    ['Correlation ID', 'Start time']
).cumcount()

# Then marking the duplicates
df.loc[(df['Prelim Call Type'] == 'Internal Transfer') & (df['Transfer Group'] > 0), 'Call Type'] = 'Duplicate Transfer'

df.drop(columns=['Prelim Call Type', 'Transfer Group'], inplace=True)

# -- Repeat Callers -- 
# Identifing repeat callers (taking into account Correlation ID across multiple days) ---
call_days = df.groupby('Correlation ID')['Date only'].nunique()
repeat_flags = call_days[call_days > 1].index
df['Is Repeat Caller'] = df['Correlation ID'].isin(repeat_flags)

# -- Call Leg Tracking --
# Creating a leg group key using only stable identifiers ---
leg_key_cols = [
    'Correlation ID',
    'Date only',
    'Start time',
    'Called number',
    'Intake Line Name',
    'PSTN vendor name'
]
df['Leg Group Key'] = df[leg_key_cols].astype(str).agg('-'.join, axis=1)

# Assigning leg numbers based on unique leg groups 
journey_groups = df.groupby(['Correlation ID', 'Date only'])

df['Leg Number'] = journey_groups['Leg Group Key'].transform(lambda x: pd.factorize(x)[0] + 1)
df['Total Legs'] = journey_groups['Leg Group Key'].transform('nunique')


In [105]:
columns_to_export = [
    'Correlation ID', 'Start time', 'Start time (Hour/Min)', 'Hour', 'Time bucket',
    'Month_Day', 'Day of week', 'Month_Year', 'Weekend', 'Business hours',
    'Direction', 'Call Type', 'Duration', 'Called number', 'Intake Line Name',
    'PSTN vendor name', 'Is Repeat Caller', 'Leg Number', 'Total Legs', 'Line Type'
]

df_export = df[columns_to_export]


In [107]:
# Accounting for duplicate rows given our specific columns
df_export = df_export.drop_duplicates()  


In [109]:
df_export.head(30)

,Correlation ID,Start time,Start time (Hour/Min),Hour,Time bucket,Month_Day,Day of week,Month_Year,Weekend,Business hours,Direction,Call Type,Duration,Called number,Intake Line Name,PSTN vendor name,Is Repeat Caller,Leg Number,Total Legs,Line Type
0,00001e73-ce33-48f1-b531-bddc7ee3965d,2024-10-05 17:30:09.903000+00:00,05:30 PM,17,5:00 PM - 5:59 PM,October 5,Saturday,October 2024,Weekend,Outside Business Hours,TERMINATING,Inbound,1961,13124312299,Migrant Legal Assistance Program,CallTower,False,1,1,Special Intake Lines
1,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 20:41:47.355000+00:00,08:41 PM,20,8:00 PM - 8:59 PM,December 11,Wednesday,December 2024,Weekday,Outside Business Hours,TERMINATING,Inbound,44,13123478311,NaN,CallTower,False,1,2,Non-Special Intake Lines
2,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 20:42:05.358000+00:00,08:42 PM,20,8:00 PM - 8:59 PM,December 11,Wednesday,December 2024,Weekday,Outside Business Hours,ORIGINATING,Internal Transfer,44,13123478300,NaN,NaN,False,2,2,Non-Special Intake Lines
3,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 20:42:05.358000+00:00,08:42 PM,20,8:00 PM - 8:59 PM,December 11,Wednesday,December 2024,Weekday,Outside Business Hours,TERMINATING,Duplicate Transfer,44,13123478300,NaN,NaN,False,2,2,Non-Special Intake Lines
4,000090ae-a71c-49e9-99d6-fdc7078dfa48,2024-09-13 19:35:32.428000+00:00,07:35 PM,19,7:00 PM - 7:59 PM,September 13,Friday,September 2024,Weekday,Outside Business Hours,ORIGINATING,Outbound,57,17086568223,NaN,CallTower,False,1,1,Non-Special Intake Lines
5,0000aef0-6f35-4560-a4be-5b3bc31fbefb,2024-11-28 15:49:47.564000+00:00,03:49 PM,15,3:00 PM - 3:59 PM,November 28,Thursday,November 2024,Weekday,Business Hours,TERMINATING,Inbound,2,13123411070,Main Number,CallTower,False,1,1,Main Number
6,0000bc93-3ace-4092-91ef-ff7cebd5ebde,2024-07-08 19:40:25.476000+00:00,07:40 PM,19,7:00 PM - 7:59 PM,July 8,Monday,July 2024,Weekday,Outside Business Hours,ORIGINATING,Internal Transfer,51,13123478302,NaN,NaN,False,1,2,Non-Special Intake Lines
7,0000bc93-3ace-4092-91ef-ff7cebd5ebde,2024-07-08 19:40:25.476000+00:00,07:40 PM,19,7:00 PM - 7:59 PM,July 8,Monday,July 2024,Weekday,Outside Business Hours,TERMINATING,Duplicate Transfer,51,13123478302,NaN,NaN,False,1,2,Non-Special Intake Lines
8,0000bc93-3ace-4092-91ef-ff7cebd5ebde,2024-07-08 19:40:43.479000+00:00,07:40 PM,19,7:00 PM - 7:59 PM,July 8,Monday,July 2024,Weekday,Outside Business Hours,TERMINATING,Internal Transfer,51,13123478300,NaN,NaN,False,2,2,Non-Special Intake Lines
9,0000bc93-3ace-4092-91ef-ff7cebd5ebde,2024-07-08 19:40:43.479000+00:00,07:40 PM,19,7:00 PM - 7:59 PM,July 8,Monday,July 2024,Weekday,Outside Business Hours,ORIGINATING,Duplicate Transfer,51,13123478300,NaN,NaN,False,2,2,Non-Special Intake Lines


In [111]:
# Exporting to CSV 

# The file name is slightly incorrect as I was having troubles changing it for Tableau purposes but file 
    # includes all calls concatenated not just intake lines -- for future, change csv name to all_calls_dashboard_ready.csv

df_export.to_csv("intake_lines_dashboard_ready.csv", index=False)
